In [ ]:
import pandas as pd
import geopandas as gpd
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv
import rasterio
from rasterio.merge import merge
from rasterio.mask import mask
import numpy as np
from shapely.geometry import mapping
from shapely import geometry
from scipy import ndimage
from shapely import wkb
from xgboost import XGBClassifier
import json
import sys
# append the path of the parent directory
sys.path.append("..")
from utils.dol import *

def binary_classification_metrics(
    df: pd.DataFrame,
    target_column: str,
    pred_column: str
):
    """
    Compute accuracy, precision (class 1), and recall (class 1)
    from a dataframe containing binary labels (0/1).

    Returns a dict with metrics.
    """

    y_true = df[target_column].astype(int)
    y_pred = df[pred_column].astype(int)

    # Confusion matrix components
    tp = ((y_true == 1) & (y_pred == 1)).sum()
    tn = ((y_true == 0) & (y_pred == 0)).sum()
    fp = ((y_true == 0) & (y_pred == 1)).sum()
    fn = ((y_true == 1) & (y_pred == 0)).sum()

    # Metrics
    accuracy = (tp + tn) / len(df) if len(df) > 0 else 0.0
    precision_1 = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall_1 = tp / (tp + fn) if (tp + fn) > 0 else 0.0

    return {
        "accuracy": accuracy,
        "precision_class_1": precision_1,
        "recall_class_1": recall_1,
        "tp": int(tp),
        "fp": int(fp),
        "fn": int(fn),
        "tn": int(tn),
    }

CACHE_DIR = "/app/data/datasets/debug/bush/cache"
MODEL_SAVE_FOLDER = "/app/data/datasets/debug/bush/models"
MODEL_SAVE_PATH = os.path.join(MODEL_SAVE_FOLDER,'dol_xgb_model_v1.json')
METADATA_SAVE_PATH = os.path.join(MODEL_SAVE_FOLDER,'dol_xgb_model_v1_metadata.json')
load_dotenv()

db_string = os.getenv('DB_STRING_PROD')
engine = create_engine(db_string)

In [ ]:
gdf_forests_zones = gpd.read_file('/app/data/datasets/debug/bush/sources/ocsge_forests_clean_types_v3.gpkg',driver='GPKG')
gdf_waters_zones = gpd.read_file('/app/data/datasets/debug/bush/sources/COURS_D_EAU.shp')
gdf_forests_zones = gdf_forests_zones[gdf_forests_zones.forest_type==1]

In [ ]:
gdf_zones_r = gpd.read_postgis("select * from detections.n_dfci_old50m_s_034_ilots", geom_col='geom',con=engine)
gdf_zones_r

In [ ]:
# reload model
xgb_model_reloaded = XGBClassifier()
xgb_model_reloaded.load_model(MODEL_SAVE_PATH)

# reload metadata
with open(METADATA_SAVE_PATH) as f:
    metadata = json.load(f)
xgb_model_reloaded.scale_pos_weight = metadata["scale_pos_weight"]
THRESHOLD = metadata["decision_threshold"]
metadata

# 1. Analysis on testset : Boissière

In [ ]:
# load : 
df_test = pd.read_parquet(os.path.join(CACHE_DIR,'test.parquet'))
y = df_test['target_control']
x = df_test.drop(columns=['target_control','sample_id'])

In [ ]:
x_test_business_features = x[['has_contact_river_zone','has_contact_forest_zone','has_inhabited_building','has_building']]
x_test_ml_features = x.drop(columns=['has_contact_river_zone','has_contact_forest_zone'])

In [ ]:
gdf_datas = gpd.read_file('/app/data/datasets/debug/bush/labels/target_dol_zones_v1.gpkg',driver='GPKG')
gdf_datas.to_crs('EPSG:2154', inplace=True)
gdf_zone_datas = gdf_datas[gdf_datas.insee_com=='34035']
gdf_zone_datas.head(5)

In [ ]:
y_proba = xgb_model_reloaded.predict_proba(x_test_ml_features)[:, 1]
 
gdf_zone_datas["proba_control"] = y_proba
gdf_zone_datas["pred_control"] = 0
gdf_zone_datas.loc[gdf_zone_datas["proba_control"] >= THRESHOLD-0.2,"pred_control"] = 1
gdf_zone_datas

In [ ]:
df_features_imp = pd.DataFrame(columns = ['features', 'importance'], data = np.asarray([x_test_ml_features.columns, xgb_model_reloaded.feature_importances_]).T)
df_features_imp.sort_values(by='importance',ascending=False).head(20)

In [ ]:
test_results_gdf = postprocess_pred_control(gdf_zone_datas, x_test_business_features)

In [ ]:
binary_classification_metrics(test_results_gdf,'target_control','pred_control_pp')

In [ ]:
test_results_gdf.to_file(
    "/app/data/datasets/debug/bush/pred_boissiere_34035_dol_zones_xgb_v1.gpkg",
    driver="GPKG"
)

# 2. analysis on unknown geozone : Puisserguier

In [ ]:
communes_test = [
    {'name':'puisserguier', 'geozone_code': 34225,'geozone_id': 297,'result_segmentation_files':  ['/app/runs/aigle_aerial_yolov_2024_puisserguier_34225_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_69.tif',
        '/app/runs/aigle_aerial_yolov_2024_puisserguier_34225_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_70.tif','/app/runs/aigle_aerial_yolov_2024_puisserguier_34225_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_83.tif',
        '/app/runs/aigle_aerial_yolov_2024_puisserguier_34225_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_84.tif']}
]
imgs_bounds = []
for comm in communes_test :
    for img_path in comm['result_segmentation_files']:
        with rasterio.open(img_path) as src :
            bbox = src.bounds
            bbox_polygon = geometry.box(*bbox)
            print(bbox_polygon)
            imgs_bounds.append([img_path, bbox_polygon])

gdf_img = gpd.GeoDataFrame(data= imgs_bounds, columns=['image_path','geometry'], geometry='geometry', crs='EPSG:2154')
gdf_img.to_crs('EPSG:2154',inplace=True)
gdf_img.drop_duplicates(subset='image_path',inplace=True)

In [ ]:
gdf_zone_data = gdf_zones_r[gdf_zones_r.insee_com.isin([str(x['geozone_code']) for x in communes_test])]
gdf_zone_data


In [ ]:
gdf_zone_data = gpd.sjoin(gdf_img,gdf_zone_data, how='right', predicate='intersects').drop(columns='index_left')
gdf_zone_data = gdf_zone_data[~gdf_zone_data.image_path.isna()]
#gdf_zone_data.rename(columns={'geom':'geometry'}, inplace=True)
gdf_zone_data

In [ ]:
x_ml_features, x_business_features =  preprocess_features(gdf_zone_data, gdf_forests_zones, gdf_waters_zones, cache_dir = CACHE_DIR, debug=False)

In [ ]:
y_proba = xgb_model_reloaded.predict_proba(x_ml_features)[:, 1]

#y_pred_custom = (y_proba >= THRESHOLD)
gdf_zone_data["proba_control"] = y_proba
gdf_zone_data["pred_control"] = 0
gdf_zone_data.loc[gdf_zone_data["proba_control"] >= THRESHOLD,"pred_control"] = 1
gdf_zone_data


In [ ]:


test_results_gdf = postprocess_pred_control(gdf_zone_data, x_business_features)

test_results_gdf.to_file(
    "/app/data/datasets/debug/bush/pred_puisserguier_34225_dol_zones_xgb_v1.gpkg",
    driver="GPKG"
)